> 🔑 **ÖĞRETMEN SÜRÜMÜ** — Bu defter, öğrenci sürümüyle aynıdır; ek olarak sonda **çözümler** bölümü vardır. Öğrencilerle bu bağlantıyı paylaşmayın.

# 🎨 LAB 3: Yapay Zekâ Ressamı — K-Means ile Renk Sanatı

> **Görev:** Bu sefer yapay zekâya **hiçbir cevap vermeyeceğiz!** Etiket yok, doğru cevap yok. Yapay zekâ verinin içindeki grupları **kendi kendine keşfedecek.**
> Önce şehre **robot şarj istasyonları** yerleştireceğiz, sonra aynı yöntemle fotoğrafları **pop-art tablolarına** dönüştüreceğiz! 🖼️

### 🤔 İki tür öğrenme
| | 👩‍🏫 Gözetimli Öğrenme (*Supervised Learning*) | 🧭 Gözetimsiz Öğrenme (*Unsupervised Learning*) |
|--|--|--|
| **Veri** | Örnekler + **doğru cevaplar** (etiketler) | Sadece örnekler, **cevap yok** |
| **Benzetme** | Öğretmenin cevap anahtarıyla çalışmak | Karışık bir LEGO kutusunu kimse söylemeden renklere göre ayırmak |
| **Örnek** | Lab 1 (rakamlar), Lab 2 (penguenler) | **Bu lab:** Kümeleme |

Benzer şeyleri bir araya toplamaya **kümeleme** (*clustering*) denir. Bugün en ünlü kümeleme algoritmasını öğreneceğiz: **K-Means** (*K-Ortalamalar*).

### 🗺️ Laboratuvar Haritası
1. 🏙️ Problem: Şarj istasyonlarını nereye kuralım?
2. 👣 K-Means'i adım adım, kendi ellerimizle çalıştıralım
3. 🔢 Kaç küme olmalı? Dirsek yöntemi
4. 🖼️ Bilgisayar renkleri nasıl görür?
5. 🎨 K-Means ile fotoğrafın renklerini azaltalım
6. 🌈 Pop-Art tablosu yapalım!
7. 📸 Kendi fotoğrafını sanat eserine çevir

> ⏱️ Süre: yaklaşık 2 ders saati &nbsp;|&nbsp; 🎮 Hücreleri sırayla **Shift + Enter** ile çalıştır.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans          # KMeans: K-Ortalamalar kümeleme algoritması
from sklearn.datasets import make_blobs     # make_blobs: deneme için kümeler halinde rastgele noktalar üretir

sns.set_theme(style="white")
np.random.seed(7)
print("Hazırız! 🎨")

---
## 1️⃣ Problem: Robot Şarj İstasyonları 🏙️🔋

Şehrimizde yüzlerce **teslimat robotu** çalışıyor. Her gün akşam robotlar bulundukları yerden en yakın şarj istasyonuna gidecek.
Belediye bize **4 tane** şarj istasyonu kurma izni verdi. **İstasyonları nereye kurmalıyız ki robotlar en az yol gitsin?**

Önce robotların akşam bulunduğu yerleri haritada görelim:

In [ ]:
# 400 robotun konumunu üretelim (gerçekte bu veriler robotların GPS'inden gelirdi)
robotlar, _ = make_blobs(n_samples=400, centers=4, cluster_std=1.3, random_state=11)   # _ → ikinci sonucu (gerçek kümeleri) kullanmayacağız!
robotlar = (robotlar - robotlar.min(axis=0)) * 0.6                                   # haritayı 0'dan başlayan kilometrelere çevirelim

plt.figure(figsize=(7, 6))
plt.scatter(robotlar[:, 0], robotlar[:, 1], s=18, c="dimgray")   # [:, 0] → tüm satırların 0. sütunu (x koordinatı)
plt.title("Akşam saatlerinde robotların konumu")
plt.xlabel("Doğu-Batı (km)")
plt.ylabel("Kuzey-Güney (km)")
plt.gca().set_aspect("equal")    # gca (get current axes = şu anki grafik): x ve y eksenlerinde 1 km aynı uzunlukta görünsün
plt.show()

> 👀 Biz insanlar bakınca robotların **birkaç bölgede toplandığını** hemen görüyoruz. Ama bilgisayara "şurası bir grup" diyen **kimse yok!** Etiket yok.
> K-Means bu grupları ve her grubun **tam merkezini** kendisi bulacak.

---
## 2️⃣ 👣 K-Means Adım Adım

K-Means algoritması çok basit 4 adımdan oluşur:

1. **🎲 Rastgele başla:** K tane istasyonu haritada rastgele yerlere koy.
2. **🧲 Grupla:** Her robot **en yakın** istasyona bağlansın.
3. **🎯 Ortala:** Her istasyonu, kendisine bağlanan robotların **tam ortasına** (ortalama konumuna) taşı. *(Algoritmanın adı buradan geliyor: K tane **ortalama**!)*
4. **🔁 Tekrarla:** İstasyonlar artık yer değiştirmeyene kadar 2. ve 3. adımları tekrar et.

Hadi bu algoritmayı **kendimiz**, sadece NumPy kullanarak yazalım!

In [ ]:
def en_yakin_istasyon(noktalar, istasyonlar):
    """Her nokta için en yakın istasyonun numarasını bulur."""
    # Her nokta ile her istasyon arasındaki mesafeyi hesapla (Pisagor teoremi! √(Δx² + Δy²))
    farklar = noktalar[:, np.newaxis, :] - istasyonlar[np.newaxis, :, :]   # newaxis: yeni bir boyut ekle → tüm çiftleri tek seferde hesapla
    mesafeler = np.sqrt((farklar ** 2).sum(axis=2))                         # sqrt (square root = karekök)
    return mesafeler.argmin(axis=1)                                          # argmin: en küçük mesafenin sıra numarası


def merkezleri_guncelle(noktalar, gruplar, k):
    """Her istasyonu, kendisine bağlı noktaların ortalamasına taşır."""
    return np.array([noktalar[gruplar == i].mean(axis=0) for i in range(k)])


K = 4
RENK_LISTESI = ["tab:red", "tab:blue", "tab:green", "tab:orange", "tab:purple", "tab:brown"]

# 1. ADIM: Rastgele başla (bilerek kötü bir başlangıç yapalım ki algoritmanın düzelttiğini görelim)
rng = np.random.default_rng(3)                                               # rng (random number generator = rastgele sayı üreteci)
istasyonlar = robotlar[rng.choice(len(robotlar), K, replace=False)]          # rastgele 4 robotun yerini başlangıç olarak seç
gecmis = [istasyonlar.copy()]                                                # istasyonların hareketlerini kaydedelim

fig, eksenler = plt.subplots(2, 2, figsize=(13, 11))
for adim, eksen in enumerate(eksenler.ravel()):
    gruplar = en_yakin_istasyon(robotlar, istasyonlar)                       # 2. ADIM: Grupla
    for i in range(K):
        eksen.scatter(*robotlar[gruplar == i].T, s=12, c=RENK_LISTESI[i], alpha=0.5)   # .T (transpose): sütunları satır yap → x ve y'yi ayır
    iz = np.array(gecmis)
    for i in range(K):
        eksen.plot(iz[:, i, 0], iz[:, i, 1], "k--", linewidth=1)            # istasyonun izlediği yol (k-- = siyah kesikli çizgi)
    eksen.scatter(*istasyonlar.T, marker="P", s=350, c=RENK_LISTESI[:K], edgecolor="black", linewidth=2)
    eksen.set_title(f"Adım {adim}" if adim else "Adım 0: Rastgele başlangıç", fontsize=13)
    eksen.set_xticks([]); eksen.set_yticks([])

    istasyonlar = merkezleri_guncelle(robotlar, gruplar, K)                  # 3. ADIM: Ortala
    gecmis.append(istasyonlar.copy())

plt.suptitle("K-Means çalışırken: İstasyonlar (+) robot gruplarının merkezine doğru kayıyor", fontsize=15)
plt.tight_layout()
plt.show()

> 🎬 **Ne oldu?** İstasyonlar her adımda kendi robot gruplarının **ortasına** doğru kaydı (kesikli çizgiler) ve birkaç adımda yerleşip durdular. Kimse bilgisayara "şu robotlar bir grup" demedi; algoritma sadece **"en yakına bağlan"** ve **"ortaya git"** kurallarıyla çalıştı. 🤯
>
> 🕵️ **Ama dikkatli bak!** Son resimde bir sorun var:
> - Alttaki robot grubu **iki istasyona** bölündü (iki istasyon yan yana, israf!) 💸
> - Sağ üstteki **iki ayrı** robot grubu ise **tek bir istasyonu** paylaşıyor ve istasyon ikisinin arasında boşlukta kaldı! 😬
>
> Neden? Çünkü **başlangıç çok kötüydü**: İki istasyon aynı grubun içinden başladı. K-Means her adımda sadece **yakın çevresine** bakarak iyileşir, bu yüzden bazen böyle "idare eder" bir çözümde **takılıp kalır.** Buna **yerel en iyi** (*local optimum* — tepeye tırmanırken asıl zirveye değil, yakındaki küçük bir tepeye çıkmak gibi) denir. ⛰️
>
> 💡 **Çözüm çok basit:** Algoritmayı **farklı rastgele başlangıçlarla birkaç kez** çalıştır, toplam yolun en kısa olduğu sonucu seç! Scikit-learn'deki `n_init` (başlangıç sayısı) ayarı tam olarak bunu yapar.
>
> 🎯 **Deney:** Yukarıdaki hücrede `default_rng(3)` içindeki 3'ü başka sayılarla (0, 1, 5, 42…) değiştirip tekrar çalıştır. Hangi başlangıçlar doğru sonuca ulaşıyor?

### ⚡ Scikit-learn ile tek satırda
Kendi yazdığımız algoritmanın aynısı Scikit-learn'de hazır. Üstelik istasyonları birbirinden uzak noktalardan başlatan **akıllı bir başlangıç yöntemi** kullanıyor ve `n_init=10` ile 10 farklı başlangıç deneyip en iyisini seçiyor.

In [ ]:
kmeans = KMeans(n_clusters=4, n_init=10, random_state=0)   # n_clusters (küme sayısı) = K, n_init: 10 farklı başlangıç dene, en iyisini seç
gruplar = kmeans.fit_predict(robotlar)                       # fit_predict: hem eğit hem de her noktanın grubunu ver

plt.figure(figsize=(7, 6))
plt.scatter(robotlar[:, 0], robotlar[:, 1], c=gruplar, cmap="tab10", s=18, alpha=0.6)
plt.scatter(*kmeans.cluster_centers_.T, marker="P", s=400, c="gold", edgecolor="black", linewidth=2, label="Şarj istasyonu")  # cluster_centers_: küme merkezleri
plt.title("Scikit-learn K-Means sonucu")
plt.xlabel("Doğu-Batı (km)")
plt.ylabel("Kuzey-Güney (km)")
plt.legend()
plt.gca().set_aspect("equal")
plt.show()

for i, (x, y) in enumerate(kmeans.cluster_centers_):
    print(f"🔋 İstasyon {i + 1}: ({x:.1f} km, {y:.1f} km) → {np.sum(gruplar == i)} robot bağlı")

---
## 3️⃣ 🔢 Kaç İstasyon Kurmalıyız? Dirsek Yöntemi

Belediye bu sefer soruyor: **"Kaç istasyon kurmamız lazım?"** 🤔
- Çok az istasyon → robotlar çok uzun yol gider. 😩
- Çok fazla istasyon → çok pahalı! 💸 (400 robota 400 istasyon kurarsak hiç yol gitmezler ama bütçe biter!)

Her K değeri için robotların istasyona olan **toplam uzaklığını** ölçelim. Bu ölçüye **eylemsizlik** (*inertia*) denir: Ne kadar küçükse gruplar o kadar sıkı.

In [ ]:
k_degerleri = range(1, 11)
toplam_uzaklik = []

for k in k_degerleri:
    model = KMeans(n_clusters=k, n_init=10, random_state=0).fit(robotlar)
    toplam_uzaklik.append(model.inertia_)     # inertia_: her noktanın kendi merkezine uzaklığının karesinin toplamı

plt.figure(figsize=(9, 4.5))
plt.plot(k_degerleri, toplam_uzaklik, marker="o", linewidth=2.5)
plt.annotate("Dirsek!", xy=(4, toplam_uzaklik[3]), xytext=(5.5, toplam_uzaklik[1]),   # annotate: oklu not ekle
             arrowprops=dict(arrowstyle="->", color="red", lw=2), fontsize=14, color="red")
plt.xlabel("K (istasyon sayısı)")
plt.ylabel("Toplam uzaklık (inertia)")
plt.title("Dirsek Yöntemi (Elbow Method)")
plt.xticks(k_degerleri)
plt.grid(alpha=0.3)
plt.show()

> 💪 **Dirsek** (*elbow*) noktası: Grafik, bükülmüş bir kola benziyor. K=4'e kadar her yeni istasyon uzaklığı **çok** azaltıyor. K=4'ten sonra yeni istasyon eklemek **pek bir şey kazandırmıyor.**
> Yani en mantıklı seçim, dirseğin olduğu yer: **4 istasyon!** Buna **dirsek yöntemi** (*elbow method*) denir.

---
## 4️⃣ 🖼️ Bilgisayar Renkleri Nasıl Görür?

Şimdi aynı algoritmayı çok farklı bir problemde kullanacağız: **Resimler!**
Bir bilgisayar ekranındaki her renk, üç ışığın karışımıdır: **R**ed (*kırmızı*), **G**reen (*yeşil*), **B**lue (*mavi*). Buna **RGB** denir.
Her ışık **0** (kapalı) ile **255** (tam açık) arasında bir değer alır:

| Renk | R | G | B |
|------|---|---|---|
| 🔴 Kırmızı | 255 | 0 | 0 |
| 🟡 Sarı | 255 | 255 | 0 |
| ⚪ Beyaz | 255 | 255 | 255 |
| ⚫ Siyah | 0 | 0 | 0 |
| 🟣 Mor | 128 | 0 | 128 |

In [ ]:
from sklearn.datasets import load_sample_image   # Scikit-learn içinde hazır gelen örnek fotoğraflar

resim = load_sample_image("china.jpg")           # Çin'de bir tapınak fotoğrafı
print("Resmin şekli (shape):", resim.shape)
print(f"→ {resim.shape[0]} satır × {resim.shape[1]} sütun piksel, her pikselde 3 sayı (R, G, B)")
print(f"→ Toplam {resim.shape[0] * resim.shape[1]:,} piksel ve {resim.size:,} sayı!")   # :, → binlik ayracı koy

plt.figure(figsize=(10, 7))
plt.imshow(resim)
plt.axis("off")
plt.title("Orijinal fotoğraf")
plt.show()

In [ ]:
# Fotoğrafın küçük bir parçasına YAKINDAN bakalım 🔍
parca = resim[196:208, 102:114]      # 12x12 piksellik bir parça kes (satırlar 196-207, sütunlar 102-113)

fig, eksen = plt.subplots(1, 2, figsize=(13, 6))
eksen[0].imshow(resim)
eksen[0].add_patch(plt.Rectangle((102, 196), 12, 12, fill=False, edgecolor="red", linewidth=3))   # Rectangle (dikdörtgen): baktığımız yeri işaretle
eksen[0].set_title("Kırmızı kare: baktığımız yer")
eksen[0].axis("off")

eksen[1].imshow(parca)
for satir in range(0, 12, 3):        # her 3 pikselden birinin RGB değerini yazalım
    for sutun in range(0, 12, 3):
        r, g, b = parca[satir, sutun]
        eksen[1].text(sutun, satir, f"{r}\n{g}\n{b}", ha="center", va="center", fontsize=7,
                      color="white" if (int(r) + int(g) + int(b)) < 380 else "black")
eksen[1].set_title("Büyütülmüş hali: her kare bir piksel (sayılar R, G, B)")
eksen[1].axis("off")
plt.show()

### 🌈 Renklerin haritası
Her pikselin 3 sayısı (R, G, B) olduğuna göre, her rengi **3 boyutlu uzayda bir nokta** gibi düşünebiliriz!
Tıpkı şarj istasyonu problemindeki robotlar gibi... Ama bu sefer harita 2 boyutlu değil, **3 boyutlu bir renk küpü!** 🧊

In [ ]:
pikseller = resim.reshape(-1, 3)          # reshape(-1, 3): resmi "her satırda bir piksel, 3 sütun (R,G,B)" olan uzun bir tabloya çevir
print("Piksel tablosunun şekli:", pikseller.shape)

ornek = pikseller[np.random.choice(len(pikseller), 4000, replace=False)]   # çizim hızlı olsun diye 4000 rastgele piksel seç

fig = plt.figure(figsize=(9, 8))
eksen = fig.add_subplot(projection="3d")   # projection="3d": 3 boyutlu grafik
eksen.scatter(ornek[:, 0], ornek[:, 1], ornek[:, 2], c=ornek / 255, s=6)   # her noktayı KENDİ rengiyle boya (renkler 0-1 arası olmalı → /255)
eksen.set_xlabel("R (kırmızı)")
eksen.set_ylabel("G (yeşil)")
eksen.set_zlabel("B (mavi)")
eksen.set_title("Fotoğraftaki her piksel, renk küpünde bir nokta")
plt.show()

> 💡 Noktalar küpün her yerine eşit dağılmamış, belli bölgelerde **kümelenmiş!** Gökyüzü mavileri bir yerde, ağaç yeşilleri başka bir yerde, tapınağın kahverengileri başka bir yerde...
>
> **İşte fikir:** Bu renk noktalarına K-Means uygularsak, fotoğrafın **en önemli K rengini** buluruz! Sonra her pikseli, kendi kümesinin merkez rengiyle boyarız. 🎨

---
## 5️⃣ 🎨 K-Means ile Renkleri Azaltalım

Bu tekniğe **renk nicemleme** (*color quantization*) denir. Eskiden bilgisayarlar sadece 16 veya 256 renk gösterebildiği için çok önemliydi. Bugün de dosyaları küçültmek ve sanatsal efektler için kullanılıyor.

In [ ]:
def renkleri_azalt(resim, k, ornek_sayisi=10_000):
    """Resmi K renge düşürür. (Yeni resim, renk paleti) döndürür."""
    pikseller = resim.reshape(-1, 3).astype(float)
    # Hız için: K-Means'i rastgele seçilmiş pikseller üzerinde eğit...
    ornek = pikseller[np.random.default_rng(0).choice(len(pikseller), min(ornek_sayisi, len(pikseller)), replace=False)]
    model = KMeans(n_clusters=k, n_init=4, random_state=0).fit(ornek)
    # ...sonra TÜM piksellerin hangi kümeye ait olduğunu tahmin et (predict)
    etiketler = model.predict(pikseller)
    palet = model.cluster_centers_.astype(np.uint8)          # uint8 (unsigned integer 8 bit): 0-255 arası tam sayı
    yeni_resim = palet[etiketler].reshape(resim.shape)       # her pikseli kendi kümesinin merkez rengiyle değiştir
    return yeni_resim, palet, etiketler.reshape(resim.shape[:2])


def paleti_goster(palet, eksen):
    """Renk paletini yan yana kareler halinde çizer."""
    eksen.imshow(palet[np.newaxis, :, :])    # paleti 1 satırlık bir resim gibi göster
    eksen.set_xticks([]); eksen.set_yticks([])


yeni, palet, _ = renkleri_azalt(resim, k=8)

fig, eksen = plt.subplots(2, 2, figsize=(15, 7.5), gridspec_kw={"height_ratios": [10, 1]})
eksen[0, 0].imshow(resim);  eksen[0, 0].set_title("Orijinal (on binlerce farklı renk)", fontsize=13)
eksen[0, 1].imshow(yeni);   eksen[0, 1].set_title("K-Means ile sadece 8 renk!", fontsize=13)
eksen[0, 0].axis("off");    eksen[0, 1].axis("off")
eksen[1, 0].axis("off")
paleti_goster(palet, eksen[1, 1])
eksen[1, 1].set_title("Yapay zekânın seçtiği 8 renk (palet)")
plt.tight_layout()
plt.show()

print(f"Orijinal fotoğraftaki farklı renk sayısı: {len(np.unique(resim.reshape(-1, 3), axis=0)):,}")

> 🤯 On binlerce renkten sadece **8 renge** indik ama fotoğraf hâlâ tanınıyor! Çünkü K-Means fotoğraf için **en önemli** 8 rengi seçti.
> Paletteki renklere bak: Gökyüzünün mavisi, ağaçların yeşili, tapınağın tonları... Hiçbirini biz söylemedik!

### 🔬 Deney: K değiştikçe ne olur?

In [ ]:
k_listesi = [2, 4, 8, 16, 32, 64]
fig, eksenler = plt.subplots(2, 3, figsize=(17, 8))

for eksen, k in zip(eksenler.ravel(), k_listesi):
    yeni, _, _ = renkleri_azalt(resim, k)
    eksen.imshow(yeni)
    eksen.set_title(f"K = {k} renk", fontsize=14)
    eksen.axis("off")
plt.suptitle("Renk sayısı arttıkça fotoğraf gerçeğe yaklaşıyor", fontsize=16)
plt.tight_layout()
plt.show()

### 💾 Bonus: Bu yöntem dosyayı ne kadar küçültür?
Bilgisayar bilgiyi **bit** (*binary digit* — ikili rakam, yani 0 ya da 1) olarak saklar.
- Orijinal resimde her piksel için 3 sayı × 8 bit = **24 bit** gerekir.
- 16 renkli bir palet kullanırsak, her piksel için sadece "kaçıncı renk?" bilgisini saklarız: 16 = 2⁴ olduğu için **4 bit** yeter!

In [ ]:
piksel_sayisi = resim.shape[0] * resim.shape[1]
orijinal_bit = piksel_sayisi * 24

print(f"{'K':>4} | {'piksel başına':>14} | {'toplam boyut':>13} | {'küçülme':>8}")
print("-" * 50)
for k in [2, 4, 16, 64, 256]:
    bit = int(np.ceil(np.log2(k)))                  # log2: 2'nin kaçıncı kuvveti? ceil (tavan): yukarı yuvarla
    toplam = piksel_sayisi * bit + k * 24           # piksellerin renk numaraları + paletin kendisi
    print(f"{k:>4} | {bit:>10} bit | {toplam / 8 / 1024:>9.1f} KB | {orijinal_bit / toplam:>6.1f}x")
print(f"\nOrijinal: {orijinal_bit / 8 / 1024:.1f} KB")

---
## 6️⃣ 🌈 Pop-Art Zamanı!

1960'larda **Andy Warhol** adlı ünlü bir sanatçı, fotoğrafları birkaç parlak renkle boyayıp yan yana dizerek **pop-art** tabloları yaptı.
Biz de yapalım! Fikir şu:
1. K-Means ile resmi **4 renk grubuna** ayır.
2. Grupları **koyudan açığa** sırala.
3. Her grubu, kendi seçtiğimiz **çılgın bir renkle** boya! 🎨

In [ ]:
def pop_art(resim, renk_paletleri):
    """Resmi, verilen her palet için farklı renklerle boyanmış pop-art karelerine çevirir."""
    k = len(renk_paletleri[0])
    _, palet, etiketler = renkleri_azalt(resim, k)
    parlaklik = palet.astype(float) @ np.array([0.299, 0.587, 0.114])   # @ : matris çarpımı. İnsan gözü yeşile daha duyarlı olduğu için ağırlıklı ortalama
    sira = np.argsort(np.argsort(parlaklik))         # argsort: sıralama. Her kümenin koyuluk sırası (0 = en koyu)
    kareler = []
    for renkler in renk_paletleri:
        renkler = np.array(renkler, dtype=np.uint8)
        kareler.append(renkler[sira[etiketler]])     # her pikseli, kümesinin koyuluk sırasına göre yeni renkle boya
    return kareler


# Her palet: en koyudan en açığa 4 renk (R, G, B)
PALETLER = [
    [(25, 25, 112), (220, 20, 60), (255, 140, 0), (255, 255, 102)],     # gece mavisi → kırmızı → turuncu → sarı
    [(48, 0, 48), (138, 43, 226), (0, 206, 209), (240, 255, 240)],      # mor → menekşe → turkuaz → açık yeşil
    [(0, 60, 0), (34, 139, 34), (255, 105, 180), (255, 240, 245)],      # koyu yeşil → yeşil → pembe → açık pembe
    [(10, 10, 10), (255, 0, 102), (0, 191, 255), (255, 255, 255)],      # siyah → fuşya → gök mavisi → beyaz
]

kareler = pop_art(resim, PALETLER)
ust = np.hstack(kareler[:2])                 # hstack (horizontal stack): resimleri yan yana yapıştır
alt = np.hstack(kareler[2:])
tablo = np.vstack([ust, alt])                # vstack (vertical stack): alt alta yapıştır

plt.figure(figsize=(14, 9.5))
plt.imshow(tablo)
plt.axis("off")
plt.title("Yapay Zekâ Pop-Art Tablosu", fontsize=18)
plt.show()

> 🎨 **Kendi paletini tasarla!** `PALETLER` listesindeki renkleri değiştir. Renk seçmek için internette **"color picker"** (*renk seçici*) diye arat, istediğin rengin RGB değerlerini kopyala.
> **Kural:** Her palette 4 renk olmalı ve **koyudan açığa** sıralanmalı (yoksa resim tuhaf görünür, ama belki de tam istediğin budur! 😄).

### 🌸 Başka bir fotoğrafla deneyelim

In [ ]:
cicek = load_sample_image("flower.jpg")
kareler = pop_art(cicek, PALETLER)

fig, eksenler = plt.subplots(1, 5, figsize=(20, 3.6))
eksenler[0].imshow(cicek)
eksenler[0].set_title("Orijinal")
for eksen, kare, no in zip(eksenler[1:], kareler, range(1, 5)):
    eksen.imshow(kare)
    eksen.set_title(f"Palet {no}")
for e in eksenler:
    e.axis("off")
plt.tight_layout()
plt.show()

---
## 7️⃣ 📸 Kendi Fotoğrafını Sanat Eserine Çevir! *(Google Colab'da çalışır)*

Hücreyi çalıştır, **"Dosya Seç"** düğmesine bas ve bir fotoğraf yükle (manzara, evcil hayvanın, robotun… 🤖🐱).
Büyük fotoğraflar otomatik olarak küçültülür, böylece işlem hızlı biter.

> 🔒 **Gizlilik uyarısı:** İnsanların yüzlerinin olduğu fotoğrafları izinleri olmadan yükleme ve paylaşma!

In [ ]:
from PIL import Image
import io

try:
    from google.colab import files       # Colab'ın dosya yükleme aracı
    yuklenen = files.upload()
    for dosya_adi, icerik in yuklenen.items():
        foto = Image.open(io.BytesIO(icerik)).convert("RGB")     # convert("RGB"): her fotoğrafı 3 kanallı RGB formatına çevir
        foto.thumbnail((640, 640))                               # thumbnail (küçük resim): en fazla 640 piksele küçült
        foto = np.array(foto)

        kareler = pop_art(foto, PALETLER)
        tablo = np.vstack([np.hstack(kareler[:2]), np.hstack(kareler[2:])])
        plt.figure(figsize=(14, 14 * tablo.shape[0] / tablo.shape[1]))
        plt.imshow(tablo)
        plt.axis("off")
        plt.title(f"{dosya_adi} — Pop-Art", fontsize=16)
        plt.show()

        Image.fromarray(tablo).save("pop_art.png")             # tabloyu dosyaya kaydet
        files.download("pop_art.png")                            # bilgisayarına indir 🎁
except ImportError:
    print("⚠️ Dosya yükleme sadece Google Colab'da çalışır. Yukarıdaki örnek fotoğraflarla deneyebilirsin!")

---
## 🏆 Meydan Okumalar (Challenge)

1. **🖍️ Boyama Kitabı:** `renkleri_azalt` ile K=2 yap. Fotoğraf bir boyama kitabı sayfasına benziyor mu? En güzel sonuç veren fotoğrafı bul!
2. **🎨 Sanat Yönetmeni:** 5 renkli bir palet tasarla. (İpucu: `pop_art` fonksiyonu palet uzunluğuna göre K'yı kendisi ayarlıyor. Ama **tüm** paletlerin uzunluğu aynı olmalı!)
3. **🏙️ Belediye Danışmanı:** Şarj istasyonu probleminde `make_blobs` içindeki `centers=4` değerini 6 yap. Dirsek yöntemi doğru sayıyı bulabiliyor mu?
4. **🧪 Bilim İnsanı:** Aynı fotoğrafa K-Means'i farklı `random_state` değerleriyle iki kez uygula. Paletler aynı mı çıkıyor? Neden olabilir?
5. **🤔 Düşünce Sorusu:** K-Means başka nerelerde kullanılabilir? (İpucu: Müşterileri alışkanlıklarına göre gruplamak, benzer şarkıları bulmak, yıldızları gruplamak…)

In [ ]:
# Meydan okumalar için kodunu buraya yaz 👇

---
## 🤔 Düşünelim: Fotoğraflar kimin?
Bu laboratuvarda fotoğrafları renklerine ayırdık ve kendi fotoğrafını yükleyebildin.

- Bir arkadaşının fotoğrafını ondan **izin almadan** bir yapay zekâ aracına yüklemek doğru mu?
- İnternette paylaşılan fotoğraflar başka yapay zekâ modellerini eğitmek için kullanılabiliyor. Sence fotoğrafın sahibine sorulmalı mı?
- K-Means bir mağazada müşterileri alışkanlıklarına göre gruplamak için de kullanılır. Hangi bilgilerin gruplamada kullanılması **rahatsız edici** olurdu?

> 🧭 **Önemli fikir:** Veri, insanlara aittir. Başkasının fotoğrafını ya da bilgisini kullanmadan önce **izin istemek**, iyi bir yapay zekâ geliştiricisinin ilk kuralıdır.

---
## 📝 Bugün Neler Öğrendik?
- 🧭 **Gözetimsiz öğrenmede** etiket yoktur. Yapay zekâ verideki yapıyı **kendisi** keşfeder.
- 🎯 **K-Means**, iki basit adımı tekrarlar: **en yakın merkeze bağlan** → **merkezi ortaya taşı.**
- 💪 **Dirsek yöntemi**, kaç küme olması gerektiğine karar vermemize yardım eder.
- 🌈 Her renk, **RGB** uzayında bir noktadır. Resimler de aslında dev sayı tablolarıdır.
- 🎨 Aynı algoritma; şarj istasyonu yerleştirmekten **dosya sıkıştırmaya** ve **sanat yapmaya** kadar çok farklı problemlerde işe yarar!

🎉 **Tebrikler, artık bir yapay zekâ ressamısın!** 🖼️🤖

---
# 🔑 Çözümler (Öğretmen İçin)
Meydan okumaların örnek çözümleri. 5. soru tartışma sorusudur.

In [ ]:
# Meydan okuma 1: Boyama Kitabı — K=2
yeni, palet, _ = renkleri_azalt(resim, k=2)
plt.figure(figsize=(6, 4)); plt.imshow(yeni); plt.axis("off"); plt.title("K = 2"); plt.show()

# Meydan okuma 2: Sanat Yönetmeni — 5 renkli paletler (hepsi 5 renk, koyudan açığa)
PALETLER_5 = [
    [(20, 20, 60), (120, 0, 120), (230, 50, 90), (255, 160, 60), (255, 240, 180)],
    [(0, 40, 40), (0, 120, 110), (80, 200, 160), (200, 240, 150), (250, 255, 230)],
]
kareler = pop_art(resim, PALETLER_5)
plt.figure(figsize=(12, 4)); plt.imshow(np.hstack(kareler)); plt.axis("off"); plt.show()

# Meydan okuma 3: Belediye Danışmanı — 6 merkez
robotlar6, _ = make_blobs(n_samples=400, centers=6, cluster_std=1.3, random_state=11)
uzakliklar = [KMeans(n_clusters=k, n_init=10, random_state=0).fit(robotlar6).inertia_ for k in range(1, 11)]
plt.plot(range(1, 11), uzakliklar, marker="o"); plt.xlabel("K"); plt.ylabel("Toplam uzaklık"); plt.show()
# Gözlem: dirsek genelde 6 civarında belirir; kümeler birbirine yakınsa dirsek belirsizleşebilir.

# Meydan okuma 4: Bilim İnsanı — farklı random_state
for rs in [0, 1]:
    pikseller = resim.reshape(-1, 3).astype(float)
    ornek = pikseller[np.random.default_rng(rs).choice(len(pikseller), 5000, replace=False)]
    m = KMeans(n_clusters=4, n_init=1, random_state=rs).fit(ornek)
    print(f"random_state={rs} palet:", m.cluster_centers_.astype(int).tolist())
# Gözlem: paletler birbirine benzer ama birebir aynı değil ve sıraları farklı olabilir:
# K-Means rastgele başlangıçla çalışır, farklı başlangıçlar farklı sonuçlara götürebilir.

---
## ✅ Defteri bitirdin mi?
Aşağıdaki hücreyi çalıştır. Önemli bölümlerin hepsini çalıştırdıysan sana bir **tamamlama kodu** verecek. Bu kodu sitede bu defterin **Tamamladım** kısmına yazınca defter tamamlanmış sayılır.

In [ ]:
# ✅ TAMAMLAMA KODU
# Bu hücre, defterin önemli bölümlerini çalıştırıp çalıştırmadığını kontrol eder.
# Hepsini çalıştırdıysan sana bir kod verir. Bu kodu Python Atölyesi sitesinde
# defterin "Tamamladım" kısmına yaz.
import hashlib
gerekenler = {
    "robotlar": "1. Problem: Robot Şarj İstasyonları",
    "kmeans": "2. K-Means Adım Adım",
    "renkleri_azalt": "5. K-Means ile Renkleri Azaltalım",
    "pop_art": "6. Pop-Art Zamanı",
    "PALETLER": "6. Pop-Art Zamanı"
}
eksikler = [bolum for ad, bolum in gerekenler.items() if ad not in globals()]
if eksikler:
    print("Henüz bitmedi! Şu bölümlerin hücrelerini çalıştırmadın:")
    for bolum in dict.fromkeys(eksikler):
        print("  •", bolum)
else:
    ozet = hashlib.sha256(("atolye|" + "|".join(gerekenler)).encode()).hexdigest()
    print("🎉 Tebrikler, defteri bitirdin! Tamamlama kodun:", ozet[:6].upper())

---
## 🔙 Python Atölyesi sitesine dön
Sitedeki **5. adım: Lab'lar** sayfasında bu laboratuvarı **Tamamladım** olarak işaretle.
Üç laboratuvarın hepsini bitirince **aşama kontrolünü** ve kursun **son testini** çözeceksin. Son testin sonunda, kursun başındaki ön testle karşılaştırılmış **gelişim raporunu** göreceksin. 📈